In [25]:
import pandas as pd
import numpy as np
from scipy.io import loadmat, savemat
import os

In [26]:
data_path = 'D:/EEG_Data_stage/datasets/'
feature_npy_file_name = 'training_features.npy'
feature_npz_file_name = 'training_features.npz'
states_file_name = 'training_scores.npy'
states_mat_name = "training_scores.mat"

## Bands
- Nx10 data array, where N is the number of epochs, and columns refer to Delta PFC, Theta HPC, Delta/Theta and EMG

In [27]:
bands = np.load(data_path + feature_npy_file_name)

In [28]:
bands.shape

(43523, 13)

## EpochsLinked
- Nx4 data array, where N is the number of epochs, and columns are described as follows:

	- column 1: epoch ID
	- column 2: epoch index (currently not used)
	- column 3: ground truth sleep stage ID, where
				- 0 is associated with artefacts,
				- 1 is associated with wakefulness,
				- 3 is associated with NREM sleep,
				- 4 is associated with TS sleep,
				- 5 is associated with REM sleep
	- column 4: the subject ID (used in multi-subject analysis only)

In [29]:
states = np.load(os.path.join(data_path, states_file_name))
timesteps = len(states)
print(timesteps)

epoch_ids = np.arange(timesteps, dtype=int)
epoch_index = np.arange(timesteps, dtype=int)

ground_truth_sleep_stage_id = states.astype(int)
ground_truth_sleep_stage_id = ground_truth_sleep_stage_id.flatten()[:timesteps]
subject_id = np.ones(timesteps, dtype=int)
epochs_linked = np.column_stack([epoch_ids, 
                                 epoch_index, 
                                 ground_truth_sleep_stage_id,
                                 subject_id
                                ])

43523


## EpochTime
- Nx3 data array, where N is the number of epochs, and columns are described as follows:

	- column 1: epoch ID
	- column 2: recording mode (i.e. baseline or recovery), where
    
			   - 1 is associated with baseline,
			   - 2 is associated with recovery (after sleep deprivation)
	- column 3: the epoch date-time

In [30]:
recording_mode = np.ones(timesteps).astype('<f8')
start_value = 41137.2918055555
step_size = 8.234392028162487e-05
time = np.arange(start_value, start_value + step_size * timesteps, step_size).astype('<f8')
time = time[:timesteps]
epoch_time = np.column_stack([epoch_ids.astype('<f8'), recording_mode, time])

## Save

In [31]:
np.savez(data_path + feature_npz_file_name, d=bands, epochsLinked=epochs_linked, epochTime=epoch_time)

In [32]:
savemat(os.path.join(data_path, states_mat_name), {"states":states})